# Kalshi Bot v2 — full pipeline

Thin notebook. All logic lives in `kalshi_v2/`. Cells here just
drive the lifecycle (build clients → start bot → inspect → stop)
and print results inline.

**Prereq.** `~/.kalshi/credentials.env` must contain:
```
KALSHI_PROD_KEY_ID=...
KALSHI_PROD_PRIVATE_KEY_PATH=~/.kalshi/prod_private_key.pem
```

**Pipeline.** Coinbase BTC spot → Kalshi WS orderbooks → empirical
bank fair value (drift-removed, vol+kurt matched) → HRDNN P-robust
filter (16 bootstrapped measures) → Lipschitz-clamped sizing →
risk preflight → place / record.

In [1]:
# Install / upgrade websockets in this kernel (v13+ uses
# additional_headers; <=10.x uses extra_headers — kalshi_v2/data.py
# auto-detects whichever is available, but v13+ is preferred).
%pip install -q 'websockets>=13' cryptography requests pandas numpy scipy python-dateutil
print('deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1')

Note: you may need to restart the kernel to use updated packages.
deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1


## 1. Setup

In [53]:
%load_ext autoreload
%autoreload 2

import time
from datetime import datetime, timezone

from kalshi_v2.config import CFG
from kalshi_v2.client import KalshiClient
from kalshi_v2 import data as v2data
from kalshi_v2.data import (fetch_historical_minutes, add_rv_features,
                              SPOT, BOOKS, TRACKED, WS_STATE, BOT_STATE)
from kalshi_v2.model import build_empirical_bank
from kalshi_v2.robust import build_ambiguity_set, SIZER
from kalshi_v2.strategy import scan_signals
from kalshi_v2.paper_db import open_trades, settled_trades
from kalshi_v2.risk import RISK_BLOCKS, get_live_balance
from kalshi_v2.portfolio import (paper_portfolio_metrics,
                                   live_portfolio_metrics, portfolio_metrics)
from kalshi_v2.main import (start_bot, stop_bot, status, kill_switch,
                              enable_live, disable_live, cancel_all_live_orders,
                              dashboard, tail_log)

print('imports OK')
print(f'mode:          {CFG["mode"]}')
print(f'live_enabled:  {CFG["live_enabled"]}')
print(f'robust_enabled: {CFG["robust_enabled"]}')
print(f'sfm_enabled:   {CFG["sfm_enabled"]}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
imports OK
mode:          live
live_enabled:  True
robust_enabled: True
sfm_enabled:   False


## 2. Build clients

Two clients: `kalshi_md` for read-only market data (no auth required
for public endpoints) and `kalshi_live` for authed actions (orders,
balance, WS handshake).

In [54]:
kalshi_md   = KalshiClient(env='prod')
kalshi_live = KalshiClient(env='prod')

print(f'md client:    base={kalshi_md.base_url}, signed={kalshi_md.private_key is not None}')
print(f'live client:  base={kalshi_live.base_url}, signed={kalshi_live.private_key is not None}')
print(f'live key_id:  {kalshi_live.key_id[:8] + "..." if kalshi_live.key_id else None}')

if kalshi_live.private_key is None:
    print('\n  ⚠ live client unsigned — bot will run paper-only, no WS auth')

md client:    base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live client:  base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live key_id:  628659b8...


In [55]:
# Quick sanity check: hit a public endpoint
try:
    ev = kalshi_md.get_events(series_ticker='KXBTC', status='open', limit=3)
    print(f'public REST OK — {len(ev.get("events", []))} BTC events open')
    for e in ev.get('events', [])[:3]:
        print(f'  {e.get("event_ticker")}')
except Exception as e:
    print(f'REST sanity failed: {e}')

if kalshi_live.private_key is not None:
    try:
        bal = kalshi_live.get_balance()
        print(f'\nlive balance: ${float(bal.get("balance", 0))/100:.2f}')
    except Exception as e:
        print(f'\nbalance fetch failed: {e}')

public REST OK — 3 BTC events open
  KXBTC-26MAY1517
  KXBTC-26MAY0917
  KXBTC-26MAY0817

live balance: $180.00


## 3. Show config

In [56]:
for k in sorted(CFG.keys()):
    v = CFG[k]
    print(f'  {k:30s} {v}')

  arb_max_dollars_per_trade      1000
  bankroll                       100000.0
  db_path                        /Users/rithvikijju/.btc_kalshi_bot/v2.db
  decision_interval_sec          5.0
  event_series                   ('KXBTC', 'KXBTCD')
  i_acknowledge_real_money_risk  False
  kalshi_fee_cap                 0.07
  lipschitz_position_L           200
  live_enabled                   True
  max_concurrent_signals         1
  max_entry_price                0.8
  max_per_market                 0.02
  max_position_age_min           90
  max_spread_cents               3
  min_edge_cents                 2.5
  min_entry_price                0.2
  min_liquidity                  0
  min_model_confidence           0.0
  mode                           live
  order_buffer_cents             2
  order_expiration_sec           30
  rest_book_interval_sec         10
  robust_enabled                 True
  robust_min_mean_edge_c         1.0
  robust_min_pass_rate           1.0
  robust_n_bootstrap

## 4. Fetch BTC history + features

90 days of BTC 1-min bars from Coinbase. This populates the input to
`build_empirical_bank` and `build_ambiguity_set`. Takes 30–60 s.

In [6]:
btc_1m = add_rv_features(fetch_historical_minutes(days_back=90))
print(f'btc_1m: {len(btc_1m):,} bars')
print(f'  range:   {btc_1m["time"].min()}  →  {btc_1m["time"].max()}')
print(f'  spot:    ${btc_1m["close"].iloc[-1]:,.0f}')
print(f'  rv_60m last:  {btc_1m["rv_60m"].iloc[-1]:.4f} (annualized)')
btc_1m.tail(3)

btc_1m: 129,171 bars
  range:   2026-02-07 19:49:00+00:00  →  2026-05-08 19:47:00+00:00
  spot:    $80,084
  rv_60m last:  0.2667 (annualized)


,time,low,high,open,close,volume,log_ret,rv_5m,rv_15m,rv_60m,rv_240m,rv_1440m
129168,2026-05-08 19:45:00+00:00,80083.57,80099.10,80093.06,80084.95,4.821030,-1.007629e-04,0.178298,0.220940,0.271134,0.318733,0.348410
129169,2026-05-08 19:46:00+00:00,80080.65,80104.73,80084.96,80085.00,4.599581,6.243368e-07,0.173607,0.212497,0.271135,0.318717,0.348271
129170,2026-05-08 19:47:00+00:00,80084.06,80084.79,80084.06,80084.06,0.074391,-1.173760e-05,0.128000,0.207805,0.266704,0.317885,0.347965


## 5. Build empirical bank — preview before bot starts

Diagnostic: same call `start_bot` will make. Confirms drift removal
and conditioning features are sane.

In [7]:
bank = build_empirical_bank(btc_1m, horizon_min=60, n_samples=5000, demean=True)
R = bank['log_returns']
import numpy as np
print(f'  n samples:     {bank["n"]:,}')
print(f'  R mean (post-demean): {R.mean():+.6f}')
print(f'  R std:         {R.std():.6f}')
print(f'  R quantiles:   1%={np.quantile(R, 0.01):+.4f}, 50%={np.quantile(R, 0.50):+.4f}, 99%={np.quantile(R, 0.99):+.4f}')
print(f'  v_mean:        {bank["v_mean"]:.4f}')
print(f'  k_mean:        {bank["k_mean"]:+.3f}')

  n samples:     5,000
  R mean (post-demean): -0.000000
  R std:         0.004858
  R quantiles:   1%=-0.0147, 50%=-0.0000, 99%=+0.0146
  v_mean:        0.4273
  k_mean:        +2.114


## 6. Build ambiguity set — preview

16 bootstrapped empirical banks. The robust filter rejects a signal
unless **every** measure agrees the trade has positive post-fee edge.
Spread of v_mean across measures shows the filter has real ambiguity
to test against (vs a near-zero spread, which would mean it's a noop).

In [8]:
amb = build_ambiguity_set(btc_1m, horizon_min=60,
                            n_bootstrap=CFG['robust_n_bootstrap'])
v_means = [m['v_mean'] for m in amb]
k_means = [m['k_mean'] for m in amb]
print(f'  measures:        {len(amb)}')
print(f'  v_mean range:   [{min(v_means):.4f}, {max(v_means):.4f}]  (spread {max(v_means)-min(v_means):.4f})')
print(f'  k_mean range:   [{min(k_means):+.3f}, {max(k_means):+.3f}]  (spread {max(k_means)-min(k_means):.3f})')
if max(v_means) - min(v_means) < 0.01:
    print('  ⚠ low spread — bootstrap may not be giving ambiguity (degenerate ambiguity set)')
else:
    print('  ✓ measures vary — filter has something to test against')

  measures:        16
  v_mean range:   [0.4110, 0.4382]  (spread 0.0272)
  k_mean range:   [+1.860, +2.291]  (spread 0.431)
  ✓ measures vary — filter has something to test against


## 7. Start the bot

Spins up 4 daemon threads:
- `spot_poller` — Coinbase BTC spot every 2 s
- `ws_listener` — Kalshi WS orderbook stream (REST fallback on 401/403)
- `event_tracker` — finds nearest BTC event in TTL window every 60 s
- `decision`     — settle → manage → scan → execute every `decision_interval_sec`

Idempotent: re-running `start_bot` while running prints a warning and
no-ops. Pass `btc_1m=btc_1m, refresh_btc_1m=False` to skip the 90-day
refetch (we already have it).

In [9]:
start_bot(kalshi_md, kalshi_live, btc_1m=btc_1m, refresh_btc_1m=False)

  ✓ empirical bank: 5000 samples
  ✓ ambiguity set: 16 measures
  ✓ live balance: $180.00

✓ bot running (4 threads). mode=paper


## 8. Live status

Re-run this cell any time to see thread health, WS state, current
tracked event, recent log lines.

In [40]:
status(last_n_log_lines=20)

  V2 BOT STATUS @ 2026-05-08T20:09:39.484208+00:00
  mode:          live
  live_enabled:  True
  running:       True
  iter:          158
  trades:        0
  live balance:  $180.00

  Threads: 4
    v2_spot_poller             alive=True
    v2_ws_listener             alive=True
    v2_event_tracker           alive=True
    v2_decision                alive=True

  WebSocket:
    mode:          websocket
    connected:     True
    subscribed:    KXBTC-26MAY0817
    msgs received: 1316
    last msg:      2026-05-08 20:09:38.056001+00:00

  Tracked event:  KXBTC-26MAY0817
  Books in mem:   50

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [20:01:32] WS connected
    [20:01:32] REST seed: 50 markets for KXBTC-26MAY0817
    [20:01:32] WS subscribed: 50 tickers × 3 channels
    [20:01:33] WS sub confirmed sid=1
    [20:01:33] WS sub confirmed sid=2
    [20:01:33] WS sub confirmed sid=3
    [20:02:02] spot err: HTTPSConnectionPool(host='api.coingecko.com', port=443): 

## 8b. Full live dashboard

Re-runnable snapshot of the entire pipeline. Shows:
- threads + WS state
- spot, sigma, tracked event, sample books
- per-market edge breakdown for every market in scope, with the reason each one was rejected (no quote, spread too wide, edge too thin, entry out of band, robust filter rejection, etc.)
- last 10 robust filter decisions
- last 10 risk-preflight blocks
- open + settled trades + realized PnL
- recent log tail

## 8c. Stream the log

`tail_log(30)` prints the last 30 lines.
`tail_log(30, follow_secs=60)` blocks and streams new entries for 60s.

In [19]:
tail_log(30)
# Or stream live for a minute:
# tail_log(30, follow_secs=60)

[19:55:05] WS connected
[19:55:05] tracking: KXBTC-26MAY0817 closes 2026-05-08 21:00:00+00:00
[19:55:05] REST seed: 50 markets for KXBTC-26MAY0817
[19:55:10] REST seed: 50 markets for KXBTC-26MAY0817
[19:55:10] WS resubscribed: 50 tickers
[19:55:10] WS sub confirmed sid=1
[19:55:11] WS sub confirmed sid=3
[19:55:11] WS sub confirmed sid=2


## 9. Manual signal scan (diagnostic)

Calls the same `scan_signals` the decision worker calls, but inline
so you can see the edge distribution and which markets passed the
robust filter.

In [20]:
from kalshi_v2.main import _EMPIRICAL_BANK, _AMBIGUITY_SET

sigs = scan_signals(empirical_bank=_EMPIRICAL_BANK,
                      ambiguity_set=_AMBIGUITY_SET)
if len(sigs) == 0:
    print('no signals this scan')
    print(f'  spot:     {SPOT.get("price")}')
    print(f'  event:    {TRACKED.get("event")}')
    print(f'  books:    {len(BOOKS)}')
else:
    cols = ['ticker', 'side', 'entry_price', 'model_p_yes', 'edge_c',
            'robust_pass_rate', 'robust_mean_edge_c', 'ttl_min']
    cols = [c for c in cols if c in sigs.columns]
    print(f'{len(sigs)} signal(s) passed all filters:\n')
    print(sigs[cols].to_string(index=False))

no signals this scan
  spot:     80101.63
  event:    KXBTC-26MAY0817
  books:    50


## 10. Open positions

In [21]:
op = open_trades()
if len(op) == 0:
    print('no open positions')
else:
    cols = ['id', 'timestamp_utc', 'market_ticker', 'side', 'contracts',
            'entry_price', 'entry_edge_cents', 'model_p_yes', 'trade_type']
    cols = [c for c in cols if c in op.columns]
    print(f'{len(op)} open positions:\n')
    print(op[cols].to_string(index=False))

no open positions


## 11. Robust filter decisions log

Every signal that came through `scan_signals` (pass or fail) is
logged here. Useful for tuning `robust_min_pass_rate` and
`robust_min_mean_edge_c`.

In [47]:
from kalshi_v2.paper_db import _conn
import pandas as pd
conn = _conn()
rd = pd.read_sql_query(
    'SELECT ts, ticker, side, entry_price, n_measures, pass_rate, '
    'mean_edge_c, min_edge_c, max_edge_c, passed '
    'FROM robust_decisions ORDER BY ts DESC LIMIT 20', conn)
conn.close()
if len(rd) == 0:
    print('no robust decisions logged yet')
else:
    print(rd.to_string(index=False))

                              ts                     ticker side  entry_price  n_measures  pass_rate  mean_edge_c  min_edge_c  max_edge_c  passed
2026-05-08T19:59:47.817602+00:00 KXBTCD-26MAY0817-T79999.99   no         0.21          16     0.9375       4.5423   -1.311867     10.8548       0


## 12. Portfolio metrics

Paper view (CFG bankroll) and live view (Kalshi balance) are kept
separate. Open positions get marked-to-market via in-memory WS
books with REST fallback.

In [48]:
portfolio_metrics(kalshi_md=kalshi_md, kalshi_live=kalshi_live)

  PAPER PORTFOLIO
  bankroll: $    100,000.00  (CFG['bankroll'])
  time:     2026-05-08T20:20:50.173323+00:00
  no trades.

  LIVE / SHADOW PORTFOLIO
  bankroll: $        180.00  (live Kalshi balance)
  time:     2026-05-08T20:20:50.175027+00:00
  no trades.



## 13. Recent settled trades

In [49]:
st = settled_trades()
if len(st) == 0:
    print('no settled trades yet')
else:
    cols = ['id', 'market_ticker', 'side', 'contracts', 'entry_price',
            'settle_price', 'pnl_dollars', 'exit_reason', 'trade_type']
    cols = [c for c in cols if c in st.columns]
    print(f'{len(st)} settled trades, last 10:\n')
    print(st[cols].tail(10).to_string(index=False))
    print(f'\ntotal realized PnL: ${st["pnl_dollars"].sum():+.2f}')
    wins = (st['pnl_dollars'] > 0).sum()
    print(f'win rate: {wins}/{len(st)} = {wins/len(st)*100:.1f}%')

no settled trades yet


## 13b. Full trade log

Every entry, every robust decision, every settlement, in chronological
order. Re-runnable any time — pulls live from the SQLite DB.

In [ ]:
from kalshi_v2.paper_db import _conn
from kalshi_v2.data import fmt_local
import pandas as pd

conn = _conn()
tr = pd.read_sql_query('SELECT * FROM trades ORDER BY timestamp_utc', conn)
rd = pd.read_sql_query('SELECT * FROM robust_decisions ORDER BY ts', conn)
conn.close()

print('=' * 78)
print(f'  TRADE LOG  ·  {len(tr)} trades  ·  {len(rd)} robust filter decisions')
print('=' * 78)

if len(tr) == 0 and len(rd) == 0:
    print('\n  no activity logged yet. The bot is waiting for a signal that')
    print('  clears the edge filter AND the robust ambiguity-set check.')
else:
    if len(rd):
        print(f'\nROBUST FILTER — {len(rd)} decisions:')
        for _, r in rd.iterrows():
            pf = '✓ PASS' if r['passed'] else '✗ REJECT'
            print(f"  {fmt_local(r['ts'])}  {pf}  {r['ticker']:32s} {r['side']:>3s}  "
                  f"entry=${r['entry_price']:.3f}  pass_rate={r['pass_rate']:.2f}  "
                  f"mean_edge={r['mean_edge_c']:+.1f}c  range=[{r['min_edge_c']:+.1f}, {r['max_edge_c']:+.1f}]")
    if len(tr):
        print(f'\nENTRIES — {len(tr)} trades:')
        for _, t in tr.iterrows():
            settled_flag = '· settled' if t['settled'] else '· open'
            ts = t['timestamp_utc'][:19] if t['timestamp_utc'] else '?'
            print(f"  #{t['id']:<3} {ts}  {t['market_ticker']:32s} {t['side']:>3s} "
                  f"x{t['contracts']} @ ${t['entry_price']:.3f}  "
                  f"edge={t['entry_edge_cents']:+.1f}c  ({t['trade_type']}) {settled_flag}")
        settled = tr[tr['settled'] == 1]
        if len(settled):
            print(f'\nSETTLEMENTS:')
            for _, t in settled.iterrows():
                print(f"  #{t['id']:<3} {fmt_local(t['settled_at']) if t['settled_at'] else '?'}  "
                      f"{t['market_ticker']:32s} settle=${t['settle_price']:.2f}  "
                      f"PnL=${t['pnl_dollars']:+.2f}  reason={t['exit_reason']}")
            wins = (settled['pnl_dollars'] > 0).sum()
            losses = (settled['pnl_dollars'] < 0).sum()
            print(f'\n  total realized PnL: ${settled["pnl_dollars"].sum():+.2f}')
            print(f'  win/loss/even:      {wins} / {losses} / {len(settled)-wins-losses}')
            if len(settled) > 0:
                print(f'  win rate:           {wins/len(settled)*100:.1f}%')

## 14. Risk-block log

Last 20 signals that were rejected by `risk_preflight`. Each entry
shows the reasons (ticker dedup, exposure cap, balance floor, etc.).

In [50]:
if not RISK_BLOCKS:
    print('no risk blocks recorded')
else:
    for rb in RISK_BLOCKS[-20:]:
        print(f'  {rb["ts"][:19]}  {rb["ticker"]:30s} {rb["side"]:>3s}  '
              f'{rb["strategy"]}  → {"; ".join(rb["reasons"])}')

no risk blocks recorded


## 15. Controls

Run any of these as needed.

In [32]:
# Stop the decision loop. Threads exit at next sleep wake (~250 ms).
# stop_bot()

# Hard kill: stop + force paper mode + refresh sessions.
# kill_switch()

# Flip mode to live. Refuses if balance unknown or $0.
enable_live()

# Flip back to paper.
# disable_live()

# Cancel every resting Kalshi order.
# cancel_all_live_orders()

✓ LIVE TRADING ENABLED. balance=$180.00


In [51]:
dashboard()

  V2 DASHBOARD @ 2026-05-08 20:20:54

A. CONNECTIVITY
   threads alive:  4/4
     ✓ v2_spot_poller
     ✓ v2_ws_listener
     ✓ v2_event_tracker
     ✓ v2_decision
   ws connected:   True  (mode=websocket, msgs=2224)
   ws subscribed:  KXBTC-26MAY0817

B. TRACKING
   spot:           $80,191.51
   causal sigma:   0.0627
   event:          KXBTC-26MAY0817
   closes:         21:00 UTC  (ttl +39.1 min)
   books in mem:   50
   sample books:
     KXBTC-26MAY0817-B65250              yes_bid=0.0  yes_ask=0.01  floor=65000
     KXBTC-26MAY0817-B65750              yes_bid=0.0  yes_ask=0.01  floor=65500
     KXBTC-26MAY0817-B66250              yes_bid=0.0  yes_ask=0.01  floor=66000

C. SCANNER (this snapshot, not the running worker)
   50 markets in event scope, 0 pass edge filters
     KXBTC-26MAY0817-T88999.99         no entry=1.000 edge=+0.0c  · edge +0.0c < 2.5c
     KXBTC-26MAY0817-B70750            no entry=1.000 edge=+0.0c  · edge +0.0c < 2.5c
     KXBTC-26MAY0817-B75250            no ent

In [58]:
status()

  V2 BOT STATUS @ 2026-05-08T20:25:22.046894+00:00
  mode:          live
  live_enabled:  True
  running:       False
  iter:          306
  trades:        0
  live balance:  $180.00

  Threads: 4
    v2_spot_poller             alive=False
    v2_ws_listener             alive=False
    v2_event_tracker           alive=False
    v2_decision                alive=False

  WebSocket:
    mode:          websocket
    connected:     False
    subscribed:    None
    msgs received: 2345
    last msg:      2026-05-08 20:22:18.081734+00:00

  Tracked event:  KXBTC-26MAY0817
  Books in mem:   50

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [20:02:06] spot err: HTTPSConnectionPool(host='api.coingecko.com', port=443): Max retries exceeded with url: /api/v3/simple/price?ids=bitcoin&vs_currencies=usd (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fbe7805b7c0>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or

In [57]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)
